# 03 — Carga no Azure SQL Server

**Squad 2 — Negócio em Tempo Real | Dupla José Vieira & Tiago Ramos**
**Tabela:** `ecommerce_itens_pedido` · **Destino:** `TABELA_DESTINO` (definido no `00`)

Grava os itens com o conector nativo `sqlserver` do Databricks; o Serverless bloqueia o JDBC genérico
para escrita. `overwrite` substitui a tabela inteira, e como o `01` relê todas as janelas, a carga é
completa e reprocessável.

| # | Regra | Como |
|---|---|---|
| L1 | Não grava DataFrame vazio nem com chave quebrada | verificações da seção 2 |
| L2 | Não encolhe o destino sem decisão explícita | `PERMITIR_REDUCAO` |
| L3 | Escrita pelo conector nativo `sqlserver` | JDBC genérico é bloqueado no Serverless |
| L4 | Carga conferida, não presumida | contagem, chave e soma de receita iguais nos dois lados |

In [0]:
%run ./01_extracao_itens_pedido

## 1. Seleção e verificações

Vão para o banco as seis colunas da origem e `dt_lote` (convertido de `timestamp_ntz` para `timestamp`,
que o conector grava como `datetime2`; o valor continua em horário de Brasília).

In [0]:
COLUNAS_DESTINO = ["id_item_pedido", "id_pedido", "sku", "quantidade", "preco_unitario", "desconto_aplicado", "dt_lote"]
df_carga = df_itens.select(*COLUNAS_DESTINO[:-1], F.col("dt_lote").cast("timestamp"))
linhas = df_carga.count()
problemas = [p for p, ruim in [("DataFrame vazio", linhas == 0),
                               (f"nulos em {CHAVE_PRIMARIA}", df_carga.filter(F.col(CHAVE_PRIMARIA).isNull()).count() > 0),
                               (f"duplicatas em {CHAVE_PRIMARIA}", linhas != df_carga.select(CHAVE_PRIMARIA).distinct().count())] if ruim]
if problemas:
    raise ValueError(f"Carga interrompida: {problemas}")
print(f"[ok] origem validada: {linhas} linhas x {len(COLUNAS_DESTINO)} colunas")

## 2. Estado atual do destino

Se o destino já tem **mais** linhas do que a origem, o `overwrite` vai encolhê-lo. Isso só segue com
`PERMITIR_REDUCAO = True`.

In [0]:
PERMITIR_REDUCAO = False
try:
    antes = spark.read.format("sqlserver").options(**opcoes_sqlserver(TABELA_DESTINO)).load().count()
    print(f"{TABELA_DESTINO} existe com {antes} linhas; origem tem {linhas}.")
except Exception as e:
    antes = None; print(f"{TABELA_DESTINO} ainda nao existe (ou esta inacessivel): sera criada. Detalhe: {type(e).__name__}")
if antes is not None and antes > linhas and not PERMITIR_REDUCAO:
    raise RuntimeError(f"O overwrite reduziria {TABELA_DESTINO} de {antes} para {linhas} linhas. Se for intencional, PERMITIR_REDUCAO = True.")

## 3. Gravação e conferência

In [0]:
df_carga.write.format("sqlserver").options(**opcoes_sqlserver(TABELA_DESTINO)).mode(MODO_ESCRITA).save()
df_destino = spark.read.format("sqlserver").options(**opcoes_sqlserver(TABELA_DESTINO)).load()

assinatura = lambda df: df.agg(F.count("*").alias("linhas"), F.countDistinct(CHAVE_PRIMARIA).alias("chaves"),
                               F.round(F.sum(F.col("preco_unitario") * F.col("quantidade")), 2).alias("receita_bruta")).first().asDict()
origem, destino = assinatura(df_carga), assinatura(df_destino)
for k in origem:
    print(f"[{'ok' if origem[k] == destino[k] else '!!'}] {k:<14} origem={origem[k]}  destino={destino[k]}")
print("Status:", "SUCESSO - origem e destino batem." if origem == destino else "ATENCAO - divergencia, investigar.")
display(df_destino.orderBy("dt_lote", CHAVE_PRIMARIA).limit(10))